In [2]:
import minimal_H as funcs
import numpy as np

Minimal 4x4 Hamiltonian Band Structure Calculation
Parameters: t = 1.0, epsilon = 0.0, a = 1.0
Brillouin Zone boundary: [-pi/(sqrt(2)*a), +pi/(sqrt(2)*a)]
k_max = 2.22144 (1/length)



In [3]:
def add_cell(H_prev, H_cell, T_hop):
    """
    Appends one unit cell H_cell to the chain H_prev coupled via T_hop.
    Automatically pads zeros for non-neighboring cells.
    """
    n_prev = H_prev.shape[0]
    n_cell = H_cell.shape[0]
    
    # Coupling block of shape (n_prev, n_cell)
    coupling = np.zeros((n_prev, n_cell), dtype=H_prev.dtype)
    coupling[-n_cell:, :] = T_hop  # Connect only to the last cell
    
    return np.block([
        [H_prev,          coupling],
        [dagger(coupling), H_cell  ]
    ])

In [4]:
dagger = lambda M: M.conj().T

t1, t2 = 1,2

def Hcell(eps, t1):
    return np.asarray([
        [eps, t1],
        [np.conjugate(t1), eps]
    ])
    
def T(t1, t2):
    return np.asarray([
        [t2, 0],
        [t1, t2]
    ])

T0 = T(t1, t2)
H0 = Hcell(0, t1)

H1 = np.block([
    [H0, T0],
    [dagger(T0), H0]
])

H2 = add_cell(H1, H0, T0)



In [5]:
def construct_H_Nblock(N, eps, t2, kx):
    t1 = funcs.calc_t1([kx,0])
    H0 = Hcell(eps, t1)
    T0 = T(t1, t2)
    H = H0
    for n in range(N):
        H = add_cell(H, H0, T0)
        
    return H
        
        

In [11]:
Hnkx = construct_H_Nblock(20, eps = 0, t2 = funcs.calc_t2([0,0]), kx = 00.01)



In [12]:
Hnkx

array([[ 0.        , -1.98595739, -0.25      , ...,  0.        ,
         0.        ,  0.        ],
       [-1.98595739,  0.        , -1.98595739, ...,  0.        ,
         0.        ,  0.        ],
       [-0.25      , -1.98595739,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
        -1.98595739, -0.25      ],
       [ 0.        ,  0.        ,  0.        , ..., -1.98595739,
         0.        , -1.98595739],
       [ 0.        ,  0.        ,  0.        , ..., -0.25      ,
        -1.98595739,  0.        ]], shape=(42, 42))